# Model Validation Architecture: Expanding Window vs. Chronological Holdout

## 1. Executive Summary & Problem Formulation
In transactional fraud detection, transactions arrive in an irreversible chronological stream characterized by **concept drift** (evolving fraud MOs), **covariate shift** (macro-spending pattern changes), and **entity churn** (continuous emergence of novel cards, devices, and IP subnets)[cite: 1]. 

Standard random $K$-fold cross-validation is fundamentally flawed for financial fraud because it causes **temporal leakage** (training on future fraud bursts to predict the past) and **entity memorization** (memorizing specific card testing spikes over a 48-hour window)[cite: 1]. In our benchmark, random partitioning produced a deceptively optimistic ROC-AUC of **~0.9417**, which collapses under production inference conditions[cite: 1].

To establish authentic production generalization, we designed two distinct temporal validation schemes tailored to the computational and architectural properties of each model family:
1. **LightGBM:** 5-Fold Expanding Window Cross-Validation (`TimeSeriesSplit`)[cite: 1].
2. **Multimodal Sequence Neural Network:** 80/20 Chronological Holdout Split[cite: 2].

---

## 2. Validation Scheme Architectures & Comparative Results

| Dimension | LightGBM Pipeline | Multimodal NN Pipeline |
| :--- | :--- | :--- |
| **Validation Strategy** | Expanding-Window `TimeSeriesSplit(n_splits=5)`[cite: 1] | Fixed Chronological Holdout (Last $20\%$)[cite: 2] |
| **Training Scope** | Progressive: $20\% \to 40\% \to 60\% \to 80\%$[cite: 1] | Fixed static: Past $80\%$ of timeline[cite: 2] |
| **Evaluation Scope** | Out-of-Fold on Folds 2–5 ($492{,}115$ samples, $83.3\%$ of dataset)[cite: 1] | Final holdout window ($118{,}108$ samples, $20.0\%$ of dataset)[cite: 2] |
| **Validation Horizon** | 4 forward rolling temporal segments[cite: 1] | 1 long-horizon forward production block[cite: 2] |
| **Primary Metric (AUC)** | **Mean CV ROC-AUC: 0.91991 ± 0.01526**[cite: 1]<br>**Overall OOF ROC-AUC: 0.91561**[cite: 1] | **Best Holdout ROC-AUC: 0.88471**[cite: 2] |
| **Leakage Defense** | Strict causal splits; zero future look-ahead[cite: 1] | Strict causal boundary; zero look-ahead[cite: 2] |

In e-commerce fraud detection, the IEEE-CIS benchmark dataset originates from Vesta Corporation, a global payment fraud prevention platform operating under a Chargeback Guarantee / Zero-Fraud-Liability model.

Unlike traditional advisory fraud APIs that simply return an informational risk score, Vesta takes on direct financial underwriting:

When the risk engine issues an APPROVE decision, Vesta guarantees the transaction.

If an approved transaction later proves fraudulent (a False Negative), the cardholder's issuing bank claws back the funds via a chargeback.

Under the guarantee contract, Vesta directly reimburses the merchant for 100% of the lost transaction amount and covers the non-refundable chargeback dispute fee (typically $15–$25 levied by payment networks like Visa and Mastercard).  
Optimizing solely for standard statistical metrics like ROC-AUC or unweighted Log-Loss is insufficient for this business model:Asymmetric Error Penalties: A False Negative costs $\text{TransactionAmt} + \text{Chargeback Fee}$, whereas a False Positive costs customer friction or manual review overhead ($2.50$). Missing a single \$500 transaction is drastically more expensive than flagging multiple small borderline transactions.Operational Queue Constraints: Routing ambiguous transactions to manual review reduces financial risk but creates strict operational capacity limits for fraud investigation teams.Card Scheme Compliance: Exceeding card network dispute thresholds (\$>0.9% for Visa / Mastercard) risks punitive multi-thousand-dollar monitoring fines and merchant processing termination.Engineering ObjectiveThis pipeline bridges the gap between offline model evaluation and real-world payment operations by:Calculating streaming rolling aggregates across 24-hour windows with zero data leakage.Deploying a calibrated, cost-sensitive 3-tier routing policy (APPROVE, MANUAL_REVIEW, DECLINE) that minimizes expected financial dollar loss rather than raw classification error.

## 3. Engineering Justification: Why the Schemes Differed

A common interview question is: *"Why didn't you run 5-fold TimeSeriesSplit on the Neural Network as well?"* 

The divergence was an intentional engineering trade-off balancing **variance estimation**, **computational tractability**, and **representation stability**:

### A. Computational Complexity of Heavy Multimodal Tracks
* **LightGBM:** Tree boosting constructs histograms over binned tabular features in minutes[cite: 1]. Running 5 folds sequentially completed in **~1.5 minutes total**, allowing exhaustive hyperparameter stability analysis across multiple market regimes[cite: 1].
* **Multimodal Neural Network:** Our architecture jointly optimizes three heavy branches[cite: 2]:
  1. A categorical embedding table stack ($17$ embedding tables)[cite: 2].
  2. A 2-layer Bidirectional/Transformer sequence encoder over 15 historical user transactions[cite: 2].
  3. A 384-dimensional frozen SentenceTransformer (`all-MiniLM-L6-v2`) semantic device embedding branch[cite: 2].
* Running a 5-fold expanding window on the NN would require repeatedly computing backward-looking stride tensors, generating device text embeddings, and running iterative backpropagation for up to 15 epochs per fold, increasing GPU training time from ~25 minutes to over **2 hours** per run with marginal diagnostic gain[cite: 2].

### B. Convergence Mechanics & Sample Efficiency
* Neural networks (specifically self-attention modules) require large data volume to learn stable representations and avoid overfitting on sparse interaction keys[cite: 2].
* In Fold 1 of `TimeSeriesSplit`, the training window contains only $98{,}425$ samples ($20\%$ of data)[cite: 1]. Training a multi-branch Transformer on only $20\%$ of noisy transaction data risks severe underfitting and divergent gradients, making early-fold scores an unrepresentative benchmark of deep learning capacity.
* The static 80/20 split provided the NN with **$472{,}432$ contiguous transactions** to learn stable positional embeddings and cross-attention correlations[cite: 2].

### C. True Production Simulation
* In real-world payment networks (e.g., Stripe, Visa), deploying a model involves training on all historical records and deploying it forward in time to predict the upcoming period. 
* The $80/20$ chronological holdout evaluates the exact production scenario: *can the model generalize to transactions occurring 1 to 2 months into the future without retraining?*

---

## 4. Empirical Performance Analysis

### Why Did LightGBM Outperform the NN (0.9156 vs. 0.8847)?
1. **Tabular Inductive Biases:** Gradient Boosted Decision Trees (GBDTs) natively excel at finding axis-aligned decision boundaries on heterogeneous, unnormalized tabular features with arbitrary numeric scales and high-cardinality frequencies[cite: 1].
2. **Temporal Window Stability:** In the LightGBM evaluation, Fold 4 and Fold 5 achieved **0.9347** and **0.9335** ROC-AUC respectively[cite: 1]. The static holdout of the NN corresponds exactly to the time period of LightGBM's Fold 5[cite: 1, 2].
3. **Sequence Sparsity:** Many transaction cards only appeared 1–2 times in the dataset, leaving the sequence Transformer padded with dummy tokens for cold-start entities[cite: 2]. LightGBM bypassed this via frequency encodings and historical target priors[cite: 1].

### Why Ensembling Still Provides Superior Alpha
Despite the NN having a lower standalone ROC-AUC, its structural error distribution differs fundamentally from LightGBM:
* **LightGBM** makes decisions based on greedy split thresholds of aggregate statistics (`mean_gain` on card velocities and transaction amounts)[cite: 1].
* **Multimodal NN** captures latent contextual relationships: sequential acceleration between transactions (via the Transformer) and subtle semantic patterns in operating systems/browsers (via Sentence-MiniLM)[cite: 2].
* Because the models leverage orthogonal inductive biases, rank-averaging their validation probabilities reduces overall prediction variance and provides an incremental lift in Out-of-Fold ROC-AUC and PR-AUC.

---

## 5. Interview Speaking Points (TL;DR)

1. **Why not random K-fold?**
   > *"Fraud detection is a non-stationary time-series problem. A random split allows temporal look-ahead leakage and entity memorization, giving an artificially inflated AUC (~0.94). Chronological splitting is non-negotiable for realistic production measurement[cite: 1]."*

2. **Why 5-fold CV for GBDT but single holdout for NN?**
   > *"LightGBM is computationally lightweight, so expanding-window TimeSeriesSplit allowed us to test stability across 4 sequential market regimes and compute metric variance (0.9199 ± 0.015)[cite: 1]. For the Multimodal NN, self-attention and dense embeddings require significant data volume to avoid underfitting, and 5-fold expanding training would have multiplied GPU compute by 5x with diminishing returns[cite: 2]. The 80/20 split matches the final production testing block[cite: 2]."*

3. **How was the ensemble evaluated fairly?**
   > *"To avoid data alignment mismatch, both models were benchmarked and blended on the identical 20% holdout partition (118,108 rows) using uniform percentile rank normalization to guard against probability calibration differences."*